# Notebook 1: Data Preprocessing and EDA

**Objective:** Transform the 600,000 records raw dataset from an "exercise-per-row" structure to a "routine-per-row" structure. This transformation handles the data aggregation challenge to ensure the correct input tensor dimensionality for the subsequent Keras DNN model.

*Execution context: This notebook should be executed locally.*

In [2]:
%pip install pandas numpy

In [3]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../') # allow imports from src folder

# Set pandas display options for easier analysis
pd.set_option('display.max_columns', None)

## 1. Load Raw Dataset

Load the `.csv` containing the exercise-level records.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Determine the base path depending on our current working directory
base_path = '../' if os.path.basename(os.getcwd()) == 'notebooks' else './'
sys.path.append(base_path) # allow imports from src folder

pd.set_option('display.max_columns', None)

In [2]:
raw_data_path = os.path.join(base_path, 'data/raw/dataset.csv')
df_raw = pd.read_csv(raw_data_path)

# Display basic statistics and shape
print(f"Initial shape: {df_raw.shape}")
display(df_raw.head())

Initial shape: (605033, 16)


,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
0,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Knee-to-wall ankle dorsiflexion test,3.0,-180.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
1,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Banded ankle distractions,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
2,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Slant board calf stretch,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
3,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Seated Tibialis Raise,3.0,15.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
4,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,90/90 Hip Rotations,3.0,8.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00


In [3]:
# Check unique values per column
print("Unique elements per column:")
display(df_raw.nunique())

# check for missing values
# 1. Count standard null values (NaN, None)
null_counts = df_raw.isnull().sum()

# 2. Count empty list representations (read as strings from CSV)
empty_list_counts = (df_raw == '[]').sum()

# 3. Combine them to see the total missing/empty values per column
total_missing = null_counts + empty_list_counts

print("Total missing or '[]' values per column:")
display(total_missing)

Unique elements per column:


title                  2598
description            2530
level                    47
goal                    250
equipment                 4
program_length           18
time_per_workout         19
week                     18
day                      10
number_of_exercises      24
exercise_name          3213
sets                     18
reps                    205
intensity                11
created                2684
last_edit               623
dtype: int64

Total missing or '[]' values per column:


title                     0
description             819
level                  2936
goal                   2936
equipment                16
program_length           16
time_per_workout          0
week                      0
day                       0
number_of_exercises       0
exercise_name             0
sets                      0
reps                      0
intensity                 0
created                  16
last_edit                41
dtype: int64

In [4]:
# Get an array of all unique exercise names
unique_exercises = df_raw['exercise_name'].dropna().unique()

# Print the total count
print(f"Total unique exercises: {len(unique_exercises)}\n")

# Display the full list
display(unique_exercises)

Total unique exercises: 3213



<StringArray>
[   'Knee-to-wall ankle dorsiflexion test',
               'Banded ankle distractions',
                'Slant board calf stretch',
                   'Seated Tibialis Raise',
                     '90/90 Hip Rotations',
   'Pigeon Stretch with Thoracic Rotation',
                           'Hip Airplanes',
             'Quadruped T-Spine Rotations',
                     'Hanging Lat Stretch',
              'Banded Shoulder Dislocates',
 ...
                'Dual Handle Lat Pulldown',
             'Decline Machine Chest Press',
        'Super ROM Lateral Dumbbell Raise',
             'Medicine Ball Russian Twist',
             'Smith Machine Reverse Lunge',
            'Super ROM Overhead Cable Row',
                     'Dumbbell Calf Jumps',
         'Rear Delt 45 Degrees Cable Flye',
 'Triceps Diverging Pressdown (Long Rope)',
                  'Core Work (You Choose)']
Length: 3213, dtype: str

In [5]:
# 1. Filter the dataset for Garage Gym
garage_gym_df = df_raw[df_raw['equipment'] == 'Garage Gym']

# 2. Extract unique exercise names from the filtered data
garage_exercises = garage_gym_df['exercise_name'].dropna().unique()

# Print the count and display the list
print(f"Total Garage Gym exercises: {len(garage_exercises)}\n")
display(garage_exercises)

Total Garage Gym exercises: 1128



<StringArray>
[ 'Knee-to-wall ankle dorsiflexion test',
             'Banded ankle distractions',
              'Slant board calf stretch',
                 'Seated Tibialis Raise',
                   '90/90 Hip Rotations',
 'Pigeon Stretch with Thoracic Rotation',
                         'Hip Airplanes',
           'Quadruped T-Spine Rotations',
                   'Hanging Lat Stretch',
            'Banded Shoulder Dislocates',
 ...
         'Bent Knee Standing Calf Raise',
                     '21 Curl (Barbell)',
                           'Waiter Curl',
          'Standing Leg Raise with Band',
                    'Leg Curl with Band',
               'Leg Extension with Band',
                             'Yates Row',
                               '1km Row',
                        'Band Push-down',
                            'Ghd Sit Up']
Length: 1128, dtype: str

In [6]:
# Create a new dataframe without 'Garage Gym' rows, leaving df_raw untouched
df_filtered = df_raw[df_raw['equipment'] != 'Garage Gym'].copy()

# Print both shapes to verify your original dataset is safe!
print(f"Original shape (df_raw): {df_raw.shape}")
print(f"Filtered shape (df_filtered): {df_filtered.shape}")

Original shape (df_raw): (605033, 16)
Filtered shape (df_filtered): (507947, 16)


In [7]:
# Check unique values per column
print("Unique elements per column:")
display(df_filtered.nunique())

# check for missing values
# 1. Count standard null values (NaN, None)
null_counts = df_filtered.isnull().sum()

# 2. Count empty list representations (read as strings from CSV)
empty_list_counts = (df_filtered == '[]').sum()

# 3. Combine them to see the total missing/empty values per column
total_missing = null_counts + empty_list_counts

print("Total missing or '[]' values per column:")
display(total_missing)

Unique elements per column:


title                  2042
description            1990
level                    46
goal                    239
equipment                 3
program_length           18
time_per_workout         19
week                     18
day                      10
number_of_exercises      24
exercise_name          2666
sets                     18
reps                    190
intensity                11
created                2107
last_edit               516
dtype: int64

Total missing or '[]' values per column:


title                     0
description             710
level                  2578
goal                   2578
equipment                16
program_length           16
time_per_workout          0
week                      0
day                       0
number_of_exercises       0
exercise_name             0
sets                      0
reps                      0
intensity                 0
created                  16
last_edit                41
dtype: int64

In [8]:
# Get an array of all unique exercise names
unique_exercises = df_filtered['exercise_name'].dropna().unique()

# Print the total count
print(f"Total unique exercises: {len(unique_exercises)}\n")

# Display the full list
display(unique_exercises)

Total unique exercises: 2666



<StringArray>
[                  'Bench Press (Barbell)',
                 'Bent Over Row (Barbell)',
                'Shoulder Press (Machine)',
                            'Lat Pulldown',
                'Tricep Extension (Cable)',
                 'Preacher Curl (Barbell)',
                         'Squat (Barbell)',
                      'Stiff Leg Deadlift',
                               'Leg Press',
                                'Leg Curl',
 ...
                'Dual Handle Lat Pulldown',
             'Decline Machine Chest Press',
        'Super ROM Lateral Dumbbell Raise',
             'Medicine Ball Russian Twist',
             'Smith Machine Reverse Lunge',
            'Super ROM Overhead Cable Row',
                     'Dumbbell Calf Jumps',
         'Rear Delt 45 Degrees Cable Flye',
 'Triceps Diverging Pressdown (Long Rope)',
                  'Core Work (You Choose)']
Length: 2666, dtype: str

In [9]:
# Count the number of rows with negative reps values
num_negative_reps = (df_filtered['reps'] < 0).sum()

# Calculate the average reps value excluding the negative values
avg_reps_non_negative = df_filtered.loc[df_filtered['reps'] >= 0, 'reps'].mean()

print(f"Number of rows with negative reps: {num_negative_reps}")
print(f"Average reps (excluding negatives): {avg_reps_non_negative}")

Number of rows with negative reps: 22204
Average reps (excluding negatives): 10.792690785044767


In [10]:
# Print each equipment type and the number of rows that use it
print(df_filtered['equipment'].value_counts())

equipment
Full Gym         464904
At Home           29934
Dumbbell Only     13093
Name: count, dtype: int64


In [12]:
# Apply the filters
df_filtered = df_filtered[
    (df_filtered['equipment'] == "Full Gym")
].copy()
# Remove the specified columns
df_final = df_filtered.drop(columns=['description', 'created', 'last_edit'])

# Display the first few rows to verify they are gone
display(df_final.head())
# Display total rows
print(f"Total rows after filtering: {len(df_final)}")
#Solo equipment Full Gym, sin reps negativos, sin level vacíos o con '[]' como string.

,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity
325,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Bench Press (Barbell),5.0,9.0,8.0
326,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Bent Over Row (Barbell),5.0,9.0,8.0
327,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Shoulder Press (Machine),3.0,14.0,8.0
328,Lyle McDonald Routine (Strength/Hypertrophy Vers),"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Full Gym,12.0,90.0,1.0,1.0,6.0,Lat Pulldown,3.0,14.0,8.0
329,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Tricep Extension (Cable),2.0,5.0,8.0


Total rows after filtering: 464904


In [17]:
# Display 10 random rows
df_final.sample(n=min(10, len(df_final)))

,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
281336,Waffle House Strong,A power-building program from Josh Bryant that...,"['Novice', 'Intermediate']","['Powerbuilding', 'Powerlifting', 'Bodybuilding']",Full Gym,7.0,60.0,2.0,2.0,5.0,Neck Flexion,3.0,10.0,8.0,2024-06-03 10:24:00,2025-06-18 08:33:00
543173,Upper Lower V3,sigma version 3,['Intermediate'],['Muscle & Sculpting'],Full Gym,16.0,110.0,6.0,3.0,13.0,SA Rear Delt Fly,2.0,20.0,6.0,2025-05-05 10:32:00,2025-06-18 09:16:00
490461,Female Fat Loss & Strength,Ajfirnfcjjckslrd,"['Advanced', 'Intermediate']",['Powerlifting'],Full Gym,12.0,40.0,2.0,2.0,6.0,Standing Calf Raise,3.0,11.0,8.0,2024-06-05 09:29:00,2025-06-18 11:48:00
41350,Noob Gains v3,"Go forward, we must. My personal 4 day split w...","['Intermediate', 'Novice']","['Bodybuilding', 'Athletics']",Full Gym,8.0,60.0,8.0,3.0,6.0,Glute-Ham Raise,3.0,8.0,8.0,2025-01-19 06:15:00,2025-06-18 07:55:00
496553,Deathstroke,A program designed to increase your physical f...,['Novice'],['Bodybuilding'],Full Gym,12.0,60.0,10.0,1.0,4.0,Mace Ball Uppercut,2.0,18.0,7.0,2024-06-08 05:22:00,2025-06-18 12:25:00
110360,Lemillion,Power! and maximal strength.,['Intermediate'],['Bodybuilding'],Full Gym,12.0,150.0,5.0,4.0,6.0,Rack Pull (Barbell),3.0,6.0,7.0,2024-02-25 05:08:00,2025-06-18 10:20:00
60299,Body Part Split Routine,To get a defined muscle.,['Novice'],['Muscle & Sculpting'],Full Gym,16.0,90.0,12.0,4.0,9.0,Overhead Press (Barbell),4.0,14.0,6.0,2024-11-27 04:10:00,2025-06-18 08:03:00
345052,Andreas general strength and fitness program,This program is made to develop basic strength...,"['Beginner', 'Novice', 'Intermediate']",['Bodybuilding'],Full Gym,10.0,60.0,9.0,2.0,7.0,Seated Shoulder Press (Dumbbell),3.0,15.0,9.0,2024-11-01 01:56:00,2025-06-18 11:29:00
434214,PPL x Arnold,Ppl x Arnold split to get massive,"['Beginner', 'Novice', 'Intermediate', 'Advanc...",['Bodybuilding'],Full Gym,8.0,60.0,2.0,3.0,7.0,Squat (Smith Machine),1.0,10.0,8.0,2025-02-23 09:24:00,2025-06-18 12:10:00
185056,Novice PL Program 1.0\n(Phoenix Powerhouse Clu...,Novice Friendly Powerlifting Program. Focusing...,"['Beginner', 'Advanced', 'Novice', 'Intermedia...",['Athletics'],Full Gym,8.0,90.0,3.0,3.0,2.0,Run,1.0,10.0,8.0,2024-11-19 09:03:00,2025-06-18 10:33:00


In [16]:
#check for null or negative values in sets, reps, intensity
import pandas as pd

# Create a summary table for the specific columns
summary_table = pd.DataFrame({
    'Null_Count': df_final[['sets', 'reps', 'intensity']].isnull().sum(),
    'Negative_Count': (df_final[['sets', 'reps', 'intensity']] < 0).sum()
})

# Display the table
display(summary_table)

,Null_Count,Negative_Count
sets,0,0
reps,0,20154
intensity,0,0


In [18]:
# Check what the negative values actually are to see if they are a code
print(df_final[df_final['reps'] < 0]['reps'].value_counts())

reps
-60.0      4153
-120.0     2165
-300.0     1427
-900.0     1349
-1800.0    1260
           ... 
-1620.0       1
-1740.0       1
-1980.0       1
-2040.0       1
-930.0        1
Name: count, Length: 122, dtype: int64


In [23]:
#Calculate volume per exercise
import numpy as np

# Calculate volume: Sets * Reps (but only if Reps >= 0. Otherwise, leave as NaN)
df_final['volume'] = np.where(
    df_final['reps'] >= 0, 
    df_final['sets'] * df_final['reps'], 
    np.nan
)

# Display the result to verify
display(df_final[['sets', 'reps', 'volume']].head())

,sets,reps,volume
325,5.0,9.0,45.0
326,5.0,9.0,45.0
327,3.0,14.0,42.0
328,3.0,14.0,42.0
329,2.0,5.0,10.0


## 2. Data Aggregation (Exercise to Routine Level)

We aggregate the dataset using custom mapping rules (e.g. summing total weight/reps for systemic load, averaging intensities). This directly builds the input vectors required for the Keras `input_shape`.

**Academic Justification:** Deep Learning models require a uniform, structured input vector for each instance. Aggregating the sequential/tabular exercise steps into a single "routine" feature vector ensures the DNN receives the global physiological snapshot of the workout (volume, load, etc.) rather than disjointed exercises.

In [ ]:
#Check if any routine has ALL exercises with missing or empty levels
import pandas as pd

def check_missing_levels_by_routine(df):
    """
    Groups the dataframe by routine ('title') and evaluates if ALL rows 
    within each routine have a missing or empty ('[]') level.
    
    Returns a dataframe with each routine and a True/False flag.
    """
    # 1. Create a boolean mask: True if level is NaN or the string representation is '[]'
    is_missing = df['level'].isna() | (df['level'].astype(str) == '[]')
    
    # 2. Attach this temporary check to the dataframe
    df_temp = df.assign(is_level_missing=is_missing)
    
    # 3. Group by title and use .all() 
    # .all() returns True ONLY if every single row in that group is True
    agg_df = df_temp.groupby('title')['is_level_missing'].all().reset_index()
    
    # Rename the column so it's easy to read
    agg_df = agg_df.rename(columns={'is_level_missing': 'all_levels_missing'})
    
    return agg_df

# Run the function
df_missing_check = check_missing_levels_by_routine(df_final)

# See the results
display(df_missing_check.head())

# Bonus: If you want to see exactly WHICH routines have completely missing levels:
completely_missing_routines = df_missing_check[df_missing_check['all_levels_missing'] == True]

print(f"\nThere are {len(completely_missing_routines)} routines where EVERY exercise is missing a level.")
if len(completely_missing_routines) > 0:
    display(completely_missing_routines)

,title,all_levels_missing
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,False
1,(NOT MY PROGRAM)SHJ Jotaro,False
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,False
3,10 week deadlift focus,False
4,1000 lbs Club,False



There are 2 routines where EVERY exercise is missing a level.


,title,all_levels_missing
986,Mania (Upper/Lower),True
1727,Viking strong,True


In [27]:
def display_routine_rows(df, routine_title):
    """
    Filters the dataframe to show only the rows that belong to a specific routine.
    """
    # Filter the dataset where the 'title' exactly matches the requested routine
    routine_df = df[df['title'] == routine_title]
    
    print(f"Showing {len(routine_df)} rows for routine: '{routine_title}'")
    
    # Display the filtered dataframe
    display(routine_df)
    
    # (Optional) Return it in case you want to save it to a variable
    return routine_df


In [ ]:
#Check the routine program 
my_routine_data = display_routine_rows(df_final, "Mania (Upper/Lower)")

Showing 288 rows for routine: 'Mania (Upper/Lower)'


,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,duration_seconds,volume
513536,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Squat (Barbell),3.0,15.0,7.0,0.0,45.0
513537,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Seated Row (Cable),2.0,12.0,7.0,0.0,24.0
513538,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Hamstring Curl,2.0,12.0,7.0,0.0,24.0
513539,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Standing Calf Raise,3.0,3.0,8.0,0.0,9.0
513540,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Leg Extension,2.0,12.0,8.0,0.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513819,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Chest Supported Row (Dumbbell),3.0,5.0,7.0,0.0,15.0
513820,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Tricep Pushdown (Cable),2.0,5.0,8.0,0.0,10.0
513821,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Incline Curl (Dumbbell),2.0,11.0,8.0,0.0,22.0
513822,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Lateral Raise (Dumbbell),2.0,9.0,7.0,0.0,18.0


## 3. Save Processed Dataset

Save the aggregated DataFrame to a CSV so it can be uploaded to Google Colab for Phase 2.

In [ ]:
# df_processed.to_csv('../data/processed/aggregated_routines.csv', index=False)
print("Preprocessing completed. Proceed to Notebook 02 on Colab.")